In [62]:
%pip install -q librosa soundfile transformers torch-geometric pandas

Note: you may need to restart the kernel to use updated packages.


In [63]:
import os
import pandas as pd


audio_folder_path = r'E:\studies\FINAL YR PROJECT\model\ICBHI_final_database'
diagnosis_file_path = r'E:\studies\FINAL YR PROJECT\model\ICBHI_final_database\important\ICBHI_Challenge_diagnosis.txt'

In [64]:
# Kaggle bootstrap: map dataset paths to this notebook's expected local layout.
import os
import glob
import shutil
from collections import Counter


def _find_icbhi_root_under_kaggle_input():
    # Preferred: explicit ICBHI_final_database folder.
    candidates = sorted(glob.glob('/kaggle/input/**/ICBHI_final_database', recursive=True))
    for path in candidates:
        if os.path.isdir(path) and len(glob.glob(os.path.join(path, '*.wav'))) > 0:
            return path

    # Fallback: infer root from diagnosis location.
    diag_paths = sorted(glob.glob('/kaggle/input/**/ICBHI_Challenge_diagnosis.txt', recursive=True))
    for diag in diag_paths:
        root = os.path.dirname(os.path.dirname(diag))
        if os.path.isdir(root) and len(glob.glob(os.path.join(root, '*.wav'))) > 0:
            return root

    # Last resort: pick directory containing the most wav files.
    wav_paths = glob.glob('/kaggle/input/**/*.wav', recursive=True)
    if len(wav_paths) == 0:
        return None

    counts = Counter(os.path.dirname(w) for w in wav_paths)
    return counts.most_common(1)[0][0]


def _create_symlink_or_copytree(src_dir, dst_dir):
    if os.path.exists(dst_dir):
        return 'exists'

    parent = os.path.dirname(os.path.abspath(dst_dir))
    if parent:
        os.makedirs(parent, exist_ok=True)

# Quick dry training run (self-contained) - inlines improved model + trainer
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
try:
    from torch_geometric.nn import GATConv, SAGEConv, BatchNorm
except Exception as e:
    print('Missing torch_geometric components:', e)
    raise

# ImprovedRespiratoryGAT (inlined)
class ImprovedRespiratoryGAT(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256, num_layers=4, num_heads=4, dropout=0.4):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.gat_layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        for i in range(num_layers):
            if i % 2 == 0:
                conv = GATConv(hidden_dim, max(1, hidden_dim // num_heads), heads=num_heads, concat=True, dropout=dropout)
            else:
                conv = SAGEConv(hidden_dim, hidden_dim)
            self.gat_layers.append(conv)
            self.norms.append(BatchNorm(hidden_dim))
        self.attention_pool = nn.Sequential(
            nn.Linear(hidden_dim, max(4, hidden_dim // 4)),
            nn.Tanh(),
            nn.Linear(max(4, hidden_dim // 4), 1),
        )
        self.wheeze_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )
        self.crackle_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )
        self.log_var_wheeze = nn.Parameter(torch.tensor(0.0))
        self.log_var_crackle = nn.Parameter(torch.tensor(0.0))
        self.dropout = dropout

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        batch = data.batch if hasattr(data, 'batch') else None
        x = self.input_proj(x)
        residuals = []
        for i, (conv, norm) in enumerate(zip(self.gat_layers, self.norms)):
            x_new = conv(x, edge_index)
            x_new = F.elu(x_new)
            x_new = norm(x_new)
            if i > 0 and i % 2 == 0:
                x_new = x_new + residuals[-1]
            x = F.dropout(x_new, p=self.dropout, training=self.training)
            residuals.append(x)
        if batch is not None:
            attn_scores = self.attention_pool(x).squeeze(-1)
            x_graph = []
            for b in torch.unique(batch):
                mask = batch == b
                scores = attn_scores[mask]
                weights = torch.softmax(scores, dim=0).unsqueeze(-1)
                x_graph.append((x[mask] * weights).sum(dim=0))
            x = torch.stack(x_graph, dim=0)
        else:
            attn_scores = self.attention_pool(x).squeeze(-1)
            weights = torch.softmax(attn_scores, dim=0).unsqueeze(-1)
            x = (x * weights).sum(dim=0, keepdim=True)
        w_logits = self.wheeze_head(x).squeeze(-1)
        c_logits = self.crackle_head(x).squeeze(-1)
        return w_logits, c_logits

# Postprocessing / calibration (inlined)
class PredictionCalibrator:
    def __init__(self, config=None):
        self.config = config or {}
        self.history = {'wheeze': [], 'crackle': []}
    def temporal_smoothing(self, predictions, window=3):
        predictions = np.asarray(predictions)
        if len(predictions) < window:
            return predictions
        smoothed = []
        half = window // 2
        for i in range(len(predictions)):
            start = max(0, i - half)
            end = min(len(predictions), i + half + 1)
            smoothed.append(np.mean(predictions[start:end]))
        return np.array(smoothed)
    def adaptive_threshold(self, predictions, base_threshold, recency_weight=0.3):
        if len(self.history['wheeze']) < 10:
            return base_threshold
        recent_fp = np.mean([1 for p in self.history['wheeze'][-10:] if p > base_threshold and p < 0.5])
        adjustment = recency_weight * recent_fp
        adjusted = min(base_threshold + adjustment, 0.85)
        return adjusted
    def minimum_duration(self, binary_predictions, min_frames=3):
        if len(binary_predictions) < min_frames:
            return np.array(binary_predictions)
        result = np.array(binary_predictions).copy()
        in_detection = False
        start_idx = 0
        for i, pred in enumerate(binary_predictions):
            if pred == 1 and not in_detection:
                in_detection = True
                start_idx = i
            elif pred == 0 and in_detection:
                in_detection = False
                if i - start_idx < min_frames:
                    result[start_idx:i] = 0
        if in_detection and len(binary_predictions) - start_idx < min_frames:
            result[start_idx:] = 0
        return result

# Focal loss + trainer (inlined)
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

def evaluate_with_postprocessing(model, loader, device, threshold=0.65, calibrator=None):
    model.eval()
    w_probs, c_probs, w_trues, c_trues = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            w_logits, c_logits = model(batch)
            w_prob = torch.sigmoid(w_logits).cpu().numpy()
            c_prob = torch.sigmoid(c_logits).cpu().numpy()
            w_probs.extend(w_prob.tolist())
            c_probs.extend(c_prob.tolist())
            if hasattr(batch, 'y_wheeze'):
                w_trues.extend(batch.y_wheeze.cpu().numpy().tolist())
            else:
                w_trues.extend([0] * len(w_prob))
            if hasattr(batch, 'y_crackle'):
                c_trues.extend(batch.y_crackle.cpu().numpy().tolist())
            else:
                c_trues.extend([0] * len(c_prob))
    w_probs = np.array(w_probs)
    c_probs = np.array(c_probs)
    w_trues = np.array(w_trues)
    c_trues = np.array(c_trues)
    if calibrator is not None:
        w_probs = calibrator.temporal_smoothing(w_probs)
        c_probs = calibrator.temporal_smoothing(c_probs)
    w_pred = (w_probs >= threshold).astype(int)
    c_pred = (c_probs >= threshold).astype(int)
    try:
        from sklearn.metrics import f1_score
    except Exception:
        f1_score = None
    if f1_score is None:
        return {'wheeze_f1': float((w_pred == w_trues).mean()), 'crackle_f1': float((c_pred == c_trues).mean()), 'weighted_f1': float(((w_pred == w_trues).mean() + (c_pred == c_trues).mean()) / 2.0)}
    w_f1 = f1_score(w_trues, w_pred, average='weighted')
    c_f1 = f1_score(c_trues, c_pred, average='weighted')
    return {'wheeze_f1': float(w_f1), 'crackle_f1': float(c_f1), 'weighted_f1': float((w_f1 + c_f1) / 2.0)}

def train_improved_model(model, train_loader, val_loader, device, epochs=60, initial_lr=0.0005, warmup_epochs=5, target_f1=0.70):
    optimizer = torch.optim.AdamW(model.parameters(), lr=initial_lr, weight_decay=0.0001)
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / max(1, warmup_epochs)
        else:
            progress = (epoch - warmup_epochs) / max(1, (epochs - warmup_epochs))
            return 0.5 * (1 + np.cos(np.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    focal_loss = FocalLoss(alpha=0.25, gamma=2.0)
    best_val_f1 = 0.0
    patience_counter = 0
    history = []
    calibrator = PredictionCalibrator(None)
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            batch = batch.to(device)
            w_logits, c_logits = model(batch)
            loss_w = focal_loss(w_logits, batch.y_wheeze.float())
            loss_c = focal_loss(c_logits, batch.y_crackle.float())
            loss = 1.25 * loss_w + loss_c
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += float(loss.item())
        scheduler.step()
        avg_loss = total_loss / max(1, len(train_loader))
        val_metrics = evaluate_with_postprocessing(model, val_loader, device, threshold=0.65, calibrator=calibrator)
        val_f1 = val_metrics.get('weighted_f1', 0.0)
        history.append({'epoch': epoch, 'train_loss': avg_loss, 'val_weighted_f1': val_f1, 'learning_rate': scheduler.get_last_lr()[0]})
        print(f'Epoch {epoch:3d}/{epochs} | Loss: {avg_loss:.4f} | Val Weighted F1: {val_f1:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}')
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), 'best_improved_respiratory_gat.pt')
            patience_counter = 0
            print(f'  ✓ New best model! Weighted F1: {best_val_f1:.4f}')
            if best_val_f1 >= target_f1:
                print(f'TARGET REACHED at epoch {epoch}! (Weighted F1 >= {target_f1})')
                break
        else:
            patience_counter += 1
            if patience_counter >= 15:
                print(f'Early stopping at epoch {epoch}')
                break
    return model, history

# Synthetic dataset with graph-level scalar labels
class SyntheticRespiratoryDataset(torch.utils.data.Dataset):
    def __init__(self, n_samples=40, frames=8, feat_dim=768, pos_prob_w=0.10, pos_prob_c=0.15):
        self.n = int(n_samples)
        self.frames = int(frames)
        self.feat_dim = int(feat_dim)
        self.pos_prob_w = float(pos_prob_w)
        self.pos_prob_c = float(pos_prob_c)
    def __len__(self):
        return self.n
    def __getitem__(self, idx):
        x = torch.randn(self.frames, self.feat_dim, dtype=torch.float32)
        edges = []
        for i in range(self.frames - 1):
            edges.append([i, i + 1])
            edges.append([i + 1, i])
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous() if len(edges) else torch.empty((2, 0), dtype=torch.long)
        y_w = torch.bernoulli(torch.tensor(self.pos_prob_w, dtype=torch.float32)).float()
        y_c = torch.bernoulli(torch.tensor(self.pos_prob_c, dtype=torch.float32)).float()
        return Data(x=x, edge_index=edge_index, y_wheeze=y_w, y_crackle=y_c)

# Build tiny datasets and loaders
train_ds = SyntheticRespiratoryDataset(n_samples=40, frames=8)
val_ds = SyntheticRespiratoryDataset(n_samples=12, frames=8)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=6, shuffle=False)

# Device from earlier notebook or fallback
device = TRAIN_DEVICE if 'TRAIN_DEVICE' in globals() else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Small model for smoke test
model = ImprovedRespiratoryGAT(input_dim=768, hidden_dim=128, num_layers=2, num_heads=4, dropout=0.2).to(device)

# Run short dry training (2 epochs)
model, history = train_improved_model(model, train_loader, val_loader, device, epochs=2, initial_lr=5e-4, warmup_epochs=1, target_f1=0.80)
print('Dry run finished. Last history entry:')
print(history[-1] if len(history) else history)

IN_KAGGLE = os.path.exists('/kaggle/input')


def resolve_audio_folder(path_hint):
    candidates = []

    if path_hint:
        candidates.append(path_hint)

    if IN_KAGGLE:
        candidates.extend([
            '/kaggle/input/datasets/catherinenassali/icbhi-final-database/ICBHI_final_database',
            '/kaggle/input/datasets/catherinenassali/icbhi-final-database',
        ])
        candidates.extend(sorted(glob.glob('/kaggle/input/**/ICBHI_final_database', recursive=True)))

    for cand in candidates:
        if os.path.isdir(cand):
            if len(glob.glob(os.path.join(cand, '*.wav'))) > 0:
                return cand
            subdirs = [d for d in glob.glob(os.path.join(cand, '*')) if os.path.isdir(d)]
            for s in subdirs:
                if len(glob.glob(os.path.join(s, '*.wav'))) > 0:
                    return s

    raise FileNotFoundError('Could not resolve an audio folder containing .wav files.')


def resolve_diagnosis_file(path_hint):
    candidates = []

    if path_hint:
        candidates.append(path_hint)

    if path_hint and os.path.isdir(path_hint):
        candidates.extend([
            os.path.join(path_hint, 'ICBHI_Challenge_diagnosis.txt'),
            os.path.join(path_hint, 'important', 'ICBHI_Challenge_diagnosis.txt'),
        ])
        candidates.extend(sorted(glob.glob(os.path.join(path_hint, '**', 'ICBHI_Challenge_diagnosis.txt'), recursive=True)))

    candidates.extend([
        'ICBHI_final_database/important/ICBHI_Challenge_diagnosis.txt',
        'ICBHI_Challenge_diagnosis.txt',
    ])

    if IN_KAGGLE:
        candidates.extend(sorted(glob.glob('/kaggle/input/**/ICBHI_Challenge_diagnosis.txt', recursive=True)))

    for cand in candidates:
        if os.path.isfile(cand):
            return cand

    raise FileNotFoundError('Could not resolve ICBHI_Challenge_diagnosis.txt from provided path hint.')


if IN_KAGGLE:
    audio_folder_hint = '/kaggle/input/datasets/catherinenassali/icbhi-final-database/ICBHI_final_database'
    diagnosis_file_hint = '/kaggle/input/datasets/catherinenassali/important'
else:
    # Prefer the absolute path if the user set `audio_folder_path` earlier in the notebook.
    audio_folder_hint = globals().get('audio_folder_path', 'ICBHI_final_database')
    diagnosis_file_hint = globals().get('diagnosis_file_path', 'ICBHI_final_database/important/ICBHI_Challenge_diagnosis.txt')

audio_folder_path = resolve_audio_folder(audio_folder_hint)
diagnosis_file_path = resolve_diagnosis_file(diagnosis_file_hint)

print('Audio folder path:', audio_folder_path)
print('Audio folder exists:', os.path.exists(audio_folder_path))
print('Diagnosis file path:', diagnosis_file_path)
print('Diagnosis file exists:', os.path.exists(diagnosis_file_path))


def load_diagnosis_file(path):
    if os.path.isdir(path):
        raise IsADirectoryError('Diagnosis path is a directory. Expected .txt file: ' + str(path))
    if not os.path.isfile(path):
        raise FileNotFoundError('Diagnosis file not found: ' + str(path))

    df = pd.read_csv(path, sep='\t', header=None, names=['patient_id', 'diagnosis'])
    df['patient_id'] = df['patient_id'].astype(str)
    return df


def scan_dataset(audio_folder):
    wav_files = sorted(glob.glob(os.path.join(audio_folder, '*.wav')))
    ann_files = sorted(glob.glob(os.path.join(audio_folder, '*.txt')))

    wav_map = {os.path.splitext(os.path.basename(f))[0]: f for f in wav_files}
    ann_map = {os.path.splitext(os.path.basename(f))[0]: f for f in ann_files}

    paired = []
    missing_ann = 0
    for base, wav_path in wav_map.items():
        if base in ann_map:
            paired.append((base, wav_path, ann_map[base]))
        else:
            missing_ann += 1

    return paired, missing_ann, len(wav_files)


def extract_patient_id(base_filename):
    return base_filename.split('_')[0]


def build_metadata_table(paired_files, diagnosis_df):
    rows = []
    diag_map = dict(zip(diagnosis_df['patient_id'], diagnosis_df['diagnosis']))

    for base, wav_path, ann_path in paired_files:
        patient_id = extract_patient_id(base)
        diagnosis = diag_map.get(patient_id, 'Unknown')
        rows.append({
            'file_id': base,
            'patient_id': patient_id,
            'diagnosis': diagnosis,
            'wav_path': wav_path,
            'ann_path': ann_path,
        })

    return pd.DataFrame(rows)


diagnosis_df = load_diagnosis_file(diagnosis_file_path)
paired_files, missing_ann, total_wavs = scan_dataset(audio_folder_path)
meta_df = build_metadata_table(paired_files, diagnosis_df)

print('\n=== Dataset Summary ===')
print('Diagnosis table shape:', diagnosis_df.shape)
print('Total wav files:', total_wavs)
print('Paired wav+annotation:', len(paired_files))
print('Missing annotation files:', missing_ann)
print('Metadata shape:', meta_df.shape)
print('\nExample pair:')
print(paired_files[0] if len(paired_files) else 'No pairs found')
print('\nDiagnosis distribution:')
print(meta_df['diagnosis'].value_counts())

Device: cpu
Epoch   0/2 | Loss: 0.1215 | Val Weighted F1: 0.7667 | LR: 0.000500
  ✓ New best model! Weighted F1: 0.7667
Epoch   1/2 | Loss: 0.1158 | Val Weighted F1: 0.7002 | LR: 0.000000
Dry run finished. Last history entry:
{'epoch': 1, 'train_loss': 0.1158088967204094, 'val_weighted_f1': 0.7002164502164501, 'learning_rate': np.float64(0.0)}
Audio folder path: E:\studies\FINAL YR PROJECT\model\ICBHI_final_database
Audio folder exists: True
Diagnosis file path: E:\studies\FINAL YR PROJECT\model\ICBHI_final_database\important\ICBHI_Challenge_diagnosis.txt
Diagnosis file exists: True

=== Dataset Summary ===
Diagnosis table shape: (126, 2)
Total wav files: 920
Paired wav+annotation: 920
Missing annotation files: 0
Metadata shape: (920, 5)

Example pair:
('101_1b1_Al_sc_Meditron', 'E:\\studies\\FINAL YR PROJECT\\model\\ICBHI_final_database\\101_1b1_Al_sc_Meditron.wav', 'E:\\studies\\FINAL YR PROJECT\\model\\ICBHI_final_database\\101_1b1_Al_sc_Meditron.txt')

Diagnosis distribution:
diagn

In [65]:
# Kaggle GPU-compat patch: safe wav2vec CPU-fallback and safe cache compute
# Inserts CPU-fallback wrappers so wav2vec inference won't crash on incompatible CUDA kernels.
import os
import json
import pickle
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from transformers import Wav2Vec2Model, Wav2Vec2Processor


def _safe_model_forward(model, input_values):
    try:
        with torch.no_grad():
            return model(input_values)
    except Exception as e:
        print("Wav2Vec forward on device failed, falling back to CPU:", repr(e))
        model_cpu = model.to("cpu")
        out = model_cpu(input_values.cpu())
        try:
            model.to(torch.device("cuda"))
        except Exception:
            pass
        return out


def wav2vec_frame_embedding_safe(frame_audio, sr=16000):
    proc = globals().get("processor", None)
    model = globals().get("wav2vec_model", None)
    if proc is None or model is None:
        proc = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
        model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")

    inputs = proc(frame_audio, sampling_rate=sr, return_tensors="pt", padding=True)
    input_values = inputs.input_values

    try:
        model_device = next(model.parameters()).device
    except StopIteration:
        model_device = torch.device("cpu")

    if model_device.type == "cuda" and torch.cuda.is_available():
        input_values = input_values.to(model_device)
    else:
        input_values = input_values.cpu()

    out = _safe_model_forward(model, input_values)
    hidden_states = out.last_hidden_state
    emb = hidden_states.mean(dim=1).squeeze(0)
    return emb.cpu().numpy().astype("float32")


# Monkeypatch Wav2Vec2FeatureCache.compute_and_cache_file for CPU fallback (if class exists)
if "Wav2Vec2FeatureCache" in globals():

    def _safe_compute_and_cache_file(self, audio_path, batch_size=32):
        cache_path = self._cache_key(audio_path)
        if os.path.exists(cache_path):
            with open(cache_path, "rb") as f:
                return pickle.load(f)

        y, _ = librosa.load(audio_path, sr=self.sr, mono=True)
        num_frames = len(y) // self.frame_samples
        frames = [
            y[i * self.frame_samples : (i + 1) * self.frame_samples]
            for i in range(num_frames)
        ]

        embeddings = []
        try:
            with torch.no_grad():
                for i in range(0, len(frames), batch_size):
                    batch = frames[i : i + batch_size]
                    inputs = self.processor(
                        batch,
                        sampling_rate=self.sr,
                        return_tensors="pt",
                        padding=True,
                    )
                    input_values = inputs.input_values.to(self._device)
                    out = self.model(input_values)
                    emb = out.last_hidden_state.mean(dim=1).cpu().numpy()
                    embeddings.append(emb)
        except Exception as e:
            print(
                "Wav2Vec2 cache compute failed on device; falling back to CPU:",
                repr(e),
            )
            model_cpu = self.model.to("cpu")
            for i in range(0, len(frames), batch_size):
                batch = frames[i : i + batch_size]
                inputs = self.processor(
                    batch,
                    sampling_rate=self.sr,
                    return_tensors="pt",
                    padding=True,
                )
                input_values = inputs.input_values.cpu()
                out = model_cpu(input_values)
                emb = out.last_hidden_state.mean(dim=1).cpu().numpy()
                embeddings.append(emb)
            try:
                self.model.to(self._device)
            except Exception:
                pass

        embeddings = (
            np.concatenate(embeddings, axis=0)
            if len(embeddings)
            else np.zeros((0, 768), dtype=np.float32)
        )
        data = {
            "embeddings": embeddings.astype(np.float32),
            "audio_duration": len(y) / self.sr,
            "num_frames": num_frames,
        }
        with open(cache_path, "wb") as f:
            pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
        return data

    Wav2Vec2FeatureCache.compute_and_cache_file = _safe_compute_and_cache_file
    print("Patched Wav2Vec2FeatureCache.compute_and_cache_file for CPU fallback.")


# Override global wav2vec_frame_embedding used elsewhere
globals()["wav2vec_frame_embedding"] = wav2vec_frame_embedding_safe
print("Inserted safe wav2vec wrapper (CPU fallback).")



Patched Wav2Vec2FeatureCache.compute_and_cache_file for CPU fallback.
Inserted safe wav2vec wrapper (CPU fallback).


In [66]:
# Inspect one batch shapes for debugging
batch_sample = next(iter(train_loader))
print('Keys:', list(batch_sample.keys()))
if hasattr(batch_sample, 'y_wheeze'):
    print('y_wheeze.shape ->', batch_sample.y_wheeze.shape)
if hasattr(batch_sample, 'y_crackle'):
    print('y_crackle.shape ->', batch_sample.y_crackle.shape)
if hasattr(batch_sample, 'batch'):
    print('batch.vector shape ->', getattr(batch_sample, 'batch').shape)
print('x.shape ->', batch_sample.x.shape)

batch_sample = batch_sample.to(TRAIN_DEVICE)
w_logits, c_logits = model(batch_sample)
print('w_logits.shape:', getattr(w_logits, 'shape', None))
print('c_logits.shape:', getattr(c_logits, 'shape', None))


Keys: ['y_crackle', 'batch', 'x', 'y_wheeze', 'edge_index', 'ptr']
y_wheeze.shape -> torch.Size([8])
y_crackle.shape -> torch.Size([8])
batch.vector shape -> torch.Size([64])
x.shape -> torch.Size([64, 768])
w_logits.shape: torch.Size([8])
c_logits.shape: torch.Size([8])


In [67]:
# Re-load improved modules robustly (search project root for `model_comparisons`)
import importlib.util, os, types, sys, pathlib


def _find_project_root(start=None, marker='model_comparisons', required_files=None, max_up=8):
    required_files = required_files or ['gnn_improved_model.py', 'train_improved.py']
    p = pathlib.Path(start or '.').resolve()
    if p.is_file():
        p = p.parent
    for _ in range(max_up + 1):
        candidate = p / marker
        if candidate.exists() and candidate.is_dir():
            # prefer marker dir that contains required files
            has_all = all((candidate / rf).exists() for rf in required_files)
            if has_all:
                return str(p)
        if p.parent == p:
            break
        p = p.parent
    # fallback: try top-level parent of start
    return str(pathlib.Path(start or '.').resolve().parent)

# Prefer an existing `proj_root` if it already points at the repo root
if 'proj_root' in globals() and os.path.isdir(proj_root) and os.path.isdir(os.path.join(proj_root, 'model_comparisons')) and os.path.isfile(os.path.join(proj_root, 'model_comparisons', 'gnn_improved_model.py')):
    _proj_root = pathlib.Path(proj_root)
else:
    # Prefer nb_dir when available (notebook directory or file), else current working dir
    start = pathlib.Path(nb_dir) if 'nb_dir' in globals() else pathlib.Path().resolve()
    _proj_root = pathlib.Path(_find_project_root(start))

proj_root = str(_proj_root)
print('Using proj_root ->', proj_root)

model_file = os.path.join(proj_root, 'model_comparisons', 'gnn_improved_model.py')
train_file = os.path.join(proj_root, 'model_comparisons', 'train_improved.py')

if not os.path.isfile(model_file):
    raise FileNotFoundError(f"Model file not found: {model_file}")

spec = importlib.util.spec_from_file_location('gnn_improved_model_live', model_file)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

# Ensure package namespace exists so sibling imports inside train_improved work
pkg = types.ModuleType('model_comparisons')
pkg.__path__ = [os.path.join(proj_root, 'model_comparisons')]
sys.modules['model_comparisons'] = pkg
sys.modules['model_comparisons.gnn_improved_model'] = mod

GATv2RespiratoryGAT = getattr(mod, 'GATv2RespiratoryGAT')
ImprovedRespiratoryGAT = getattr(mod, 'ImprovedRespiratoryGAT')

spec2 = importlib.util.spec_from_file_location('train_improved_live', train_file)
mod2 = importlib.util.module_from_spec(spec2)
spec2.loader.exec_module(mod2)
train_improved_model = getattr(mod2, 'train_improved_model')

print('Re-loaded updated modules')


Using proj_root -> E:\studies\FINAL YR PROJECT\model
Re-loaded updated modules


In [68]:
# Start full training (30 epochs)
import time, traceback, importlib.util, os, sys
print('Starting full training (30 epochs) ...')
start_time = time.time()

try:
    # Ensure train_improved_model available
    try:
        from model_comparisons.train_improved import train_improved_model
    except Exception:
        spec = importlib.util.spec_from_file_location(
            'train_improved_live', os.path.join(proj_root, 'model_comparisons', 'train_improved.py')
        )
        mod_train = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod_train)
        train_improved_model = getattr(mod_train, 'train_improved_model')

    # Ensure model classes available
    try:
        from model_comparisons.gnn_improved_model import GATv2RespiratoryGAT, ImprovedRespiratoryGAT
    except Exception:
        mod_gnn = sys.modules.get('model_comparisons.gnn_improved_model')
        if mod_gnn is None:
            spec2 = importlib.util.spec_from_file_location(
                'gnn_improved_model_live', os.path.join(proj_root, 'model_comparisons', 'gnn_improved_model.py')
            )
            mod_gnn = importlib.util.module_from_spec(spec2)
            spec2.loader.exec_module(mod_gnn)
            sys.modules['model_comparisons.gnn_improved_model'] = mod_gnn
        GATv2RespiratoryGAT = getattr(mod_gnn, 'GATv2RespiratoryGAT')
        ImprovedRespiratoryGAT = getattr(mod_gnn, 'ImprovedRespiratoryGAT')

    # instantiate model (prefer GATv2)
    try:
        model_full = GATv2RespiratoryGAT(input_dim=768, hidden_dim=256, num_layers=4, num_heads=4, dropout=0.2).to(TRAIN_DEVICE)
        print('Using GATv2RespiratoryGAT')
    except Exception as e:
        print('GATv2 instantiation failed, falling back to ImprovedRespiratoryGAT:', e)
        model_full = ImprovedRespiratoryGAT(input_dim=768, hidden_dim=256, num_layers=4, num_heads=4, dropout=0.2).to(TRAIN_DEVICE)

    epochs = 30
    save_path = 'best_gatv2_full.pt'

    model_full, history_full = train_improved_model(
        model=model_full,
        train_loader=train_loader,
        val_loader=val_loader,
        device=TRAIN_DEVICE,
        epochs=epochs,
        initial_lr=5e-4,
        warmup_epochs=1,
        target_f1=0.65,
        config=None,
        attention_l2=1e-5,
        use_uncertainty=True,
        wheeze_pos_weight=w_pw,
        crackle_pos_weight=c_pw,
        use_focal=False,
        focal_gamma=2.0,
        save_path=save_path,
    )

    print('Full training complete. Saved:', save_path)
    print('History entries:', len(history_full))

except Exception as e:
    traceback.print_exc()
    print('Training failed:', e)

finally:
    print('Elapsed (s):', round(time.time() - start_time, 1))


Starting full training (30 epochs) ...
Using GATv2RespiratoryGAT
Epoch   0/60 | Loss: 1.5726 | Val Weighted F1: 0.8788 | LR: 0.000200
  ✓ New best model! Weighted F1: 0.8788
TARGET REACHED at epoch 0! (Weighted F1 >= 0.65)
Full training complete. Saved: best_gatv2_full.pt
History entries: 1
Elapsed (s): 0.4


In [69]:
# Quick shape test: run GATv2 forward on one batch
batch_sample = next(iter(train_loader))
print('batch sizes -> nodes:', batch_sample.x.shape[0], 'graphs:', batch_sample.y_wheeze.shape[0])

batch_sample = batch_sample.to(TRAIN_DEVICE)
try:
    test_model = GATv2RespiratoryGAT(input_dim=768, hidden_dim=256, num_layers=4, num_heads=4, dropout=0.2).to(TRAIN_DEVICE)
    w_t, c_t = test_model(batch_sample)
    print('GATv2RespiratoryGAT outputs -> w:', getattr(w_t,'shape',None), 'c:', getattr(c_t,'shape',None))
except Exception as e:
    import traceback; traceback.print_exc()
    print('GATv2 forward failed:', e)

try:
    test_model2 = ImprovedRespiratoryGAT(input_dim=768, hidden_dim=256, num_layers=4, num_heads=4, dropout=0.2).to(TRAIN_DEVICE)
    w2, c2 = test_model2(batch_sample)
    print('ImprovedRespiratoryGAT outputs -> w:', getattr(w2,'shape',None), 'c:', getattr(c2,'shape',None))
except Exception as e:
    import traceback; traceback.print_exc()
    print('ImprovedRespiratoryGAT forward failed:', e)


batch sizes -> nodes: 64 graphs: 8
GATv2RespiratoryGAT outputs -> w: torch.Size([8]) c: torch.Size([8])
ImprovedRespiratoryGAT outputs -> w: torch.Size([8]) c: torch.Size([8])


In [70]:
# Debug: inspect torch.unique(batch) and manual attention pooling for GATv2
batch = next(iter(train_loader))
print('raw nodes:', batch.x.shape[0], 'graphs:', batch.y_wheeze.shape[0])

batch = batch.to(TRAIN_DEVICE)
print('batch vector shape:', batch.batch.shape, 'dtype:', batch.batch.dtype)
unique = torch.unique(batch.batch)
print('unique values:', unique, 'count:', unique.numel())

# instantiate small GATv2 for debug (reuse test_model if present)
try:
    gm = GATv2RespiratoryGAT(input_dim=768, hidden_dim=256, num_layers=4, num_heads=4, dropout=0.2).to(TRAIN_DEVICE)
except Exception:
    gm = model_full

x = gm.input_proj(batch.x)
attn_scores = gm.attention_pool(x).squeeze(-1)
print('x.shape after proj:', x.shape, 'attn_scores.shape:', attn_scores.shape)

x_graph = []
for b in unique:
    mask = (batch.batch == b)
    print('b', int(b.item()), 'mask_sum', int(mask.sum().item()))
    scores = attn_scores[mask]
    weights = torch.softmax(scores, dim=0).unsqueeze(-1)
    pooled = (x[mask] * weights).sum(dim=0)
    print(' pooled shape:', pooled.shape)
    x_graph.append(pooled)

x_graph_stacked = torch.stack(x_graph, dim=0)
print('x_graph_stacked.shape:', x_graph_stacked.shape)


raw nodes: 64 graphs: 8
batch vector shape: torch.Size([64]) dtype: torch.int64
unique values: tensor([0, 1, 2, 3, 4, 5, 6, 7]) count: 8
x.shape after proj: torch.Size([64, 256]) attn_scores.shape: torch.Size([64])
b 0 mask_sum 8
 pooled shape: torch.Size([256])
b 1 mask_sum 8
 pooled shape: torch.Size([256])
b 2 mask_sum 8
 pooled shape: torch.Size([256])
b 3 mask_sum 8
 pooled shape: torch.Size([256])
b 4 mask_sum 8
 pooled shape: torch.Size([256])
b 5 mask_sum 8
 pooled shape: torch.Size([256])
b 6 mask_sum 8
 pooled shape: torch.Size([256])
b 7 mask_sum 8
 pooled shape: torch.Size([256])
x_graph_stacked.shape: torch.Size([8, 256])


In [71]:
# Full training: ImprovedRespiratoryGAT (30 epochs)
print("Starting ImprovedRespiratoryGAT full training (30 epochs)...")

from model_comparisons.train_improved import train_improved_model
from model_comparisons.gnn_improved_model import ImprovedRespiratoryGAT

save_path = "best_improved_30epochs.pt"

# Ensure we have a usable ImprovedRespiratoryGAT instance
try:
    is_ok = isinstance(model, ImprovedRespiratoryGAT)
except Exception:
    is_ok = False

if not is_ok:
    print('`model` is not an ImprovedRespiratoryGAT instance; creating a new one')
    model = ImprovedRespiratoryGAT(input_dim=768, hidden_dim=256, num_layers=4, num_heads=4, dropout=0.4)

model = model.to(TRAIN_DEVICE)

model_trained, history_trained = train_improved_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=TRAIN_DEVICE,
    epochs=30,
    save_path=save_path,
)

print('Training finished, saved:', save_path)
trained_model = model_trained
training_history = history_trained


Starting ImprovedRespiratoryGAT full training (30 epochs)...
`model` is not an ImprovedRespiratoryGAT instance; creating a new one


Epoch   0/60 | Loss: 0.0889 | Val Weighted F1: 0.4881 | LR: 0.000200
  ✓ New best model! Weighted F1: 0.4881
Epoch   1/60 | Loss: 0.0927 | Val Weighted F1: 0.5937 | LR: 0.000300
  ✓ New best model! Weighted F1: 0.5937
Epoch   2/60 | Loss: 0.0972 | Val Weighted F1: 0.7002 | LR: 0.000400
  ✓ New best model! Weighted F1: 0.7002
TARGET REACHED at epoch 2! (Weighted F1 >= 0.7)
Training finished, saved: best_improved_30epochs.pt


In [72]:
# Re-run full training: force 30 epochs (disable early stopping)
print("Re-running ImprovedRespiratoryGAT training for full 30 epochs (no early stop)...")

from model_comparisons.train_improved import train_improved_model
from model_comparisons.gnn_improved_model import ImprovedRespiratoryGAT

save_path = "best_improved_30epochs_no_earlystop.pt"

# Create a fresh model to avoid any checkpoint/optimizer state
model = ImprovedRespiratoryGAT(input_dim=768, hidden_dim=256, num_layers=4, num_heads=4, dropout=0.4)
model = model.to(TRAIN_DEVICE)

config = {"training": {"epochs": 30, "learning_rate": 0.0005, "warmup_epochs": 5}}

model_trained, history_trained = train_improved_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=TRAIN_DEVICE,
    epochs=30,
    config=config,
    target_f1=999.0,  # effectively disable early stopping via target
    save_path=save_path,
)

print('Training finished, saved:', save_path)
trained_model_full = model_trained
training_history_full = history_trained


Re-running ImprovedRespiratoryGAT training for full 30 epochs (no early stop)...
Epoch   0/30 | Loss: 0.1262 | Val Weighted F1: 0.8788 | LR: 0.000200
  ✓ New best model! Weighted F1: 0.8788
Epoch   1/30 | Loss: 0.0955 | Val Weighted F1: 0.8768 | LR: 0.000300
Epoch   2/30 | Loss: 0.0952 | Val Weighted F1: 0.8788 | LR: 0.000400
Epoch   3/30 | Loss: 0.0983 | Val Weighted F1: 0.8768 | LR: 0.000500
Epoch   4/30 | Loss: 0.0950 | Val Weighted F1: 0.6667 | LR: 0.000500
Epoch   5/30 | Loss: 0.0679 | Val Weighted F1: 0.9384 | LR: 0.000498
  ✓ New best model! Weighted F1: 0.9384
Epoch   6/30 | Loss: 0.0968 | Val Weighted F1: 0.8214 | LR: 0.000492
Epoch   7/30 | Loss: 0.0734 | Val Weighted F1: 0.8768 | LR: 0.000482
Epoch   8/30 | Loss: 0.0725 | Val Weighted F1: 0.8172 | LR: 0.000469
Epoch   9/30 | Loss: 0.0586 | Val Weighted F1: 0.8788 | LR: 0.000452
Epoch  10/30 | Loss: 0.0484 | Val Weighted F1: 0.7002 | LR: 0.000432
Epoch  11/30 | Loss: 0.0445 | Val Weighted F1: 0.9384 | LR: 0.000409
Epoch  12/3